### Pin classification with EfficientNet-B4 fine-tune


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import timm
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

BACKEND = 'efficientnet_b4'
MODEL_ID = 'tf_efficientnet_b4.ns_jft_in1k'
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "pin_classification"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "places365_pin_manifest.parquet",
    PROJECT_ROOT / "data" / "processed" / "food_101_pin_manifest.parquet",
    PROJECT_ROOT / "data" / "processed" / "inaturalist_pin_manifest.parquet",
]

sns.set_theme(style="whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
def compute_multiclass_metrics(y_true, probas, classes) -> dict:
    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
    from sklearn.preprocessing import label_binarize

    y_true = np.asarray(y_true)
    probas = np.asarray(probas)
    pred_idx = probas.argmax(axis=1)
    pred_labels = np.asarray(classes)[pred_idx]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred_labels, average="macro", zero_division=0
    )

    y_bin = label_binarize(y_true, classes=classes)
    try:
        auc = roc_auc_score(y_bin, probas, average="macro", multi_class="ovr")
    except ValueError:
        auc = float("nan")

    fpr_values = []
    for class_index, class_name in enumerate(classes):
        y_pos = (y_true == class_name).astype(int)
        pred_pos = (pred_labels == class_name).astype(int)
        fp = int(((pred_pos == 1) & (y_pos == 0)).sum())
        tn = int(((pred_pos == 0) & (y_pos == 0)).sum())
        fpr_values.append(fp / max(fp + tn, 1))

    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(np.mean(fpr_values)), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
    }


def load_pin_frame() -> pd.DataFrame:
    for candidate in PROCESSED_CANDIDATES:
        if candidate.exists():
            frame = pd.read_parquet(candidate)
            frame["source_dataset"] = candidate.stem.replace("_pin_manifest", "")
            return frame
    raise FileNotFoundError("Run one of the pin dataset notebooks first.")


pin_df = load_pin_frame()
pin_df["image_paths"] = pin_df["image_paths_json"].map(json.loads)
classes = sorted(pin_df["label_name"].unique().tolist())
class_to_idx = {label: idx for idx, label in enumerate(classes)}

train_df, valid_df = train_test_split(
    pin_df,
    test_size=0.2 if len(pin_df) >= 100 else 0.3,
    stratify=pin_df["label_name"] if pin_df["label_name"].nunique() > 1 else None,
    random_state=42,
)

transform = transforms.Compose(
    [
        transforms.Resize((380, 380) if BACKEND == "efficientnet_b4" else (224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


class PinDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform, max_images: int = 4):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.max_images = max_images

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        images = []
        for path in row["image_paths"][: self.max_images]:
            images.append(self.transform(Image.open(path).convert("RGB")))
        images = torch.stack(images, dim=0)
        return images, class_to_idx[row["label_name"]], row["pin_id"]


def collate_pins(batch):
    image_tensors, labels, pin_ids = zip(*batch)
    return list(image_tensors), torch.tensor(labels), list(pin_ids)


train_loader = DataLoader(PinDataset(train_df, transform), batch_size=4, shuffle=True, collate_fn=collate_pins)
valid_loader = DataLoader(PinDataset(valid_df, transform), batch_size=4, shuffle=False, collate_fn=collate_pins)


In [ ]:
if BACKEND == "places365_resnet50":
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    ckpt_path = ARTIFACT_ROOT / "resnet50_places365.pth.tar"
    if ckpt_path.exists():
        checkpoint = torch.load(ckpt_path, map_location="cpu")
        state_dict = checkpoint.get("state_dict", checkpoint)
        state_dict = {key.replace("module.", ""): value for key, value in state_dict.items()}
        model.load_state_dict(state_dict, strict=False)
    model.fc = nn.Linear(model.fc.in_features, len(classes))
else:
    model = timm.create_model(MODEL_ID, pretrained=True, num_classes=len(classes))
model = model.to(device)

sample_images, _, _ = next(iter(valid_loader))
with torch.no_grad():
    logits = model(sample_images[0].to(device))
print("Per-image logits shape:", tuple(logits.shape))

feature_maps = {}

def save_hook(name):
    def hook(_, __, output):
        feature_maps[name] = output.detach().cpu()
    return hook

layer_handle = getattr(model, "layer4", getattr(model, "conv_head", None))
if layer_handle is not None and hasattr(layer_handle, "register_forward_hook"):
    handle = layer_handle.register_forward_hook(save_hook("features"))
    with torch.no_grad():
        _ = model(sample_images[0].to(device))
    handle.remove()
    activation = feature_maps["features"][0]
    if activation.ndim == 3:
        activation = activation.mean(dim=0)
    else:
        activation = activation.mean(dim=0)
    plt.figure(figsize=(6, 4))
    sns.heatmap(activation.numpy(), cmap="mako")
    plt.title("Intermediate activation summary")
    plt.tight_layout()


In [ ]:
def compute_multiclass_metrics(y_true, probas, classes) -> dict:
    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
    from sklearn.preprocessing import label_binarize

    y_true = np.asarray(y_true)
    probas = np.asarray(probas)
    pred_idx = probas.argmax(axis=1)
    pred_labels = np.asarray(classes)[pred_idx]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred_labels, average="macro", zero_division=0
    )

    y_bin = label_binarize(y_true, classes=classes)
    try:
        auc = roc_auc_score(y_bin, probas, average="macro", multi_class="ovr")
    except ValueError:
        auc = float("nan")

    fpr_values = []
    for class_index, class_name in enumerate(classes):
        y_pos = (y_true == class_name).astype(int)
        pred_pos = (pred_labels == class_name).astype(int)
        fp = int(((pred_pos == 1) & (y_pos == 0)).sum())
        tn = int(((pred_pos == 0) & (y_pos == 0)).sum())
        fpr_values.append(fp / max(fp + tn, 1))

    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(np.mean(fpr_values)), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
    }


def pin_forward(batch_images: list[torch.Tensor]) -> torch.Tensor:
    pin_logits = []
    for image_stack in batch_images:
        logits = model(image_stack.to(device))
        pin_logits.append(logits.mean(dim=0))
    return torch.stack(pin_logits, dim=0)


def evaluate_model() -> tuple[dict, np.ndarray]:
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch_images, labels, _ in valid_loader:
            logits = pin_forward(batch_images)
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
            all_labels.extend([classes[idx] for idx in labels.numpy().tolist()])
    all_probs = np.vstack(all_probs)
    return compute_multiclass_metrics(all_labels, all_probs, classes), all_probs


optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
history = []
for epoch in range(2):
    model.train()
    running_loss = 0.0
    for batch_images, labels, _ in train_loader:
        optimizer.zero_grad()
        logits = pin_forward(batch_images)
        loss = criterion(logits, labels.to(device))
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(labels)
    metrics, _ = evaluate_model()
    metrics["epoch"] = epoch + 1
    metrics["loss"] = round(running_loss / max(len(train_df), 1), 4)
    history.append(metrics)

history_df = pd.DataFrame(history)
history_df


In [ ]:
run_dir = ARTIFACT_ROOT / ("pin_" + BACKEND)
run_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), run_dir / "model.pt")
(run_dir / "history.json").write_text(history_df.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")

final_metrics, final_probs = evaluate_model()
pd.DataFrame(final_probs, columns=classes).to_parquet(run_dir / "validation_probas.parquet", index=False)
(run_dir / "metrics.json").write_text(json.dumps(final_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
final_metrics
